### Inference Model with NMS

In [1]:
import os, json, cv2
import glob
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.evaluation.coco_evaluation import instances_to_coco_json

In [2]:
import warnings
warnings.simplefilter("ignore")

In [ ]:
# ----------------------------
# 1️⃣ Load config & model
# ----------------------------
trained_model_dir = './models/250525_R50_RESTOR_fold_0'
config_file = os.path.join(trained_model_dir, 'config.yaml')
model_path = os.path.join(trained_model_dir, 'model_best_AP50.pth')

cfg = get_cfg()
cfg.merge_from_file(config_file)
cfg.MODEL.WEIGHTS = model_path
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST = 0.3

predictor = DefaultPredictor(cfg)

Loading config ./trained-model/250525_R50_RESTOR_fold_0\config.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


In [4]:
# ----------------------------
# 2️⃣ Folder input/output
# ----------------------------
imgs_dir = './sample/tiles/'
pred_dir = os.path.join(imgs_dir, 'predictions/')
os.makedirs(pred_dir, exist_ok=True)

# ----------------------------
# 3️⃣ Loop setiap gambar
# ----------------------------
print("Predicting...")
for img_path in glob.glob(os.path.join(imgs_dir, '*.tif')):
    base_name = os.path.basename(img_path)
    file_name, _ = os.path.splitext(base_name)
    
    # Load and predict
    im = cv2.imread(img_path)
    outputs = predictor(im)
    
    # Simpan hasil ke JSON
    output_file = os.path.join(pred_dir, f"{file_name}.json")
    predict_img = instances_to_coco_json(outputs["instances"].to("cpu"), img_path)
    with open(output_file, "w") as f:
        json.dump(predict_img, f)
    print(f"{output_file}")

print("\nPrediction complete!")

Predicting...
./sample/tiles/predictions/tile_0_0.json
./sample/tiles/predictions/tile_0_1.json
./sample/tiles/predictions/tile_0_10.json
./sample/tiles/predictions/tile_0_11.json
./sample/tiles/predictions/tile_0_12.json
./sample/tiles/predictions/tile_0_2.json
./sample/tiles/predictions/tile_0_3.json
./sample/tiles/predictions/tile_0_4.json
./sample/tiles/predictions/tile_0_5.json
./sample/tiles/predictions/tile_0_6.json
./sample/tiles/predictions/tile_0_7.json
./sample/tiles/predictions/tile_0_8.json
./sample/tiles/predictions/tile_0_9.json
./sample/tiles/predictions/tile_10_0.json
./sample/tiles/predictions/tile_10_1.json
./sample/tiles/predictions/tile_10_10.json
./sample/tiles/predictions/tile_10_11.json
./sample/tiles/predictions/tile_10_12.json
./sample/tiles/predictions/tile_10_2.json
./sample/tiles/predictions/tile_10_3.json
./sample/tiles/predictions/tile_10_4.json
./sample/tiles/predictions/tile_10_5.json
./sample/tiles/predictions/tile_10_6.json
./sample/tiles/predictions/

# Prediction to LabelMe JSON

In [7]:
pred_dir

'./sample/tiles/predictions/'

In [8]:
import json
import os
import glob
import numpy as np
from skimage import measure
import pycocotools.mask as mask_util # Penting untuk decode RLE
import sys

# ==============================================================================
# FUNGSI HELPER 
# ==============================================================================

def close_contour(contour):
    if not np.array_equal(contour[0], contour[-1]):
        contour = np.vstack((contour, contour[0]))
    return contour

def binary_mask_to_polygon(binary_mask, tolerance=0):
    """ 
    Mengonversi binary mask 2D tunggal menjadi daftar poligon format COCO.
    """
    polygons = []
    if binary_mask.ndim != 2:
        raise ValueError("binary_mask_to_polygon expects a 2D mask")
        
    padded_binary_mask = np.pad(binary_mask, pad_width=1, mode='constant', constant_values=0)
    contours = measure.find_contours(padded_binary_mask, 0.5)

    for contour in contours:
        contour = close_contour(contour)
        approx_polygon = measure.approximate_polygon(contour, tolerance)
        if len(approx_polygon) < 3:
            continue
        approx_polygon = np.flip(approx_polygon, axis=1)
        segmentation = [item for sublist in approx_polygon for item in sublist]
        segmentation = [0 if i < 0 else i for i in segmentation]
        polygons.append(segmentation)

    return polygons # Mengembalikan list of polygons, cth: [[x1,y1...], [x3,y3...]]

# ==============================================================================
# FUNGSI KONVERSI KE LABELME
# ==============================================================================

def polygons_to_labelme_json(detections_data_list, image_name, height, width, label="tree"):
    """
    Mengonversi daftar poligon datar gaya COCO menjadi struktur JSON LabelMe.
    """
    shapes = []

    # Loop melalui list data deteksi
    # 'det_data' akan berisi {"polygon": [x1,y1...], "score": 0.99}
    for det_data in detections_data_list:
        poly_flat = det_data["polygon"] # Ambil data poligon | poly_flat adalah [x1, y1, x2, y2, ...]
        score = det_data["score"]       # Ambil data skor

        points = []
        try:
            for i in range(0, len(poly_flat), 2):
                points.append([float(poly_flat[i]), float(poly_flat[i+1])])
        except IndexError:
            print(f"  Warning: Skipping malformed polygon in {image_name}.")
            continue
            
        if len(points) < 3:
            continue

        shapes.append({
            "label": label,
            "score": score,
            "points": points,
            "group_id": None,
            "description": "",
            "shape_type": "polygon",
            "flags": {}
        })

    labelme_json = {
        "version": "2.5.3",
        "flags": {},
        "shapes": shapes,
        "imagePath": image_name,
        "imageData": None,
        "imageHeight": height,
        "imageWidth": width
    }
    return labelme_json

In [ ]:
# SCRIPT UTAMA
def main_individual_rle_converter():
    
    # 1. TENTUKAN PATH ANDA
    prediction_dir = pred_dir
    
    # Direktori output untuk menyimpan file LabelMe
    #output_labelme_dir = imgs_dir
    output_labelme_dir = os.path.join(prediction_dir, "labelme_output")
    os.makedirs(output_labelme_dir, exist_ok=True)
    
    # Cari semua file .json di direktori prediksi
    # 'recursive=False' agar tidak mencari di dalam 'labelme_converted_output'
    json_files_to_process = glob.glob(os.path.join(prediction_dir, "*.json"), recursive=False)
    
    print(f"🚀 Memulai konversi (Mode Individual)...")
    print(f"Mendeteksi {len(json_files_to_process)} file .json di: {prediction_dir}")
    print(f"Menyimpan file LabelMe ke: {output_labelme_dir}")
    
    count = 0
    total = len(json_files_to_process)
    
    # 2. ITERASI SETIAP FILE JSON
    for i, json_file_path in enumerate(json_files_to_process):
        file_basename = os.path.basename(json_file_path)
            
        print(f"[{i+1}/{total}] Memproses: {file_basename}")

        # 3. BACA ISI FILE JSON
        try:
            with open(json_file_path, 'r') as f:
                rle_detections = json.load(f) # Ini adalah DAFTAR (list) deteksi

            if not isinstance(rle_detections, list) or not rle_detections:
                print(f"  Warning: File {file_basename} kosong atau formatnya salah. Dilewati.")
                continue
                
        except Exception as e:
            print(f"  Warning: Gagal membaca {file_basename}. Detail: {e}. Dilewati.")
            continue

        # 4. EKSTRAK DATA DAN DECODE RLE
        all_detections_data_for_this_image = []
        
        try:
            # Ambil info gambar dari deteksi pertama di dalam file
            first_det = rle_detections[0]
            height, width = first_det["segmentation"]["size"]
            # Ambil nama file gambar asli dari 'image_id'
            image_name = os.path.basename(first_det["image_id"])
        except Exception as e:
            print(f"  Warning: Format data di {file_basename} tidak terduga. Dilewati. ({e})")
            continue

        # Loop untuk setiap deteksi (pohon) di dalam file JSON ini
        for det_idx, det in enumerate(rle_detections):
            score = det.get("score")
            rle = det["segmentation"]
            try:
                binary_mask_2d = mask_util.decode(rle).astype(np.uint8)
            except Exception as e:
                print(f"  Warning: Gagal decode RLE #{det_idx+1} di {file_basename}. Detail: {e}")
                continue
            
            # Konversi mask ke polygon
            polygons = binary_mask_to_polygon(binary_mask_2d)

            # Simpan setiap poligon BERSAMA dengan skor aslinya
            for poly_data in polygons:
                all_detections_data_for_this_image.append({
                    "polygon": poly_data,
                    "score": score
                })

        if not all_detections_data_for_this_image:
            print(f"  Info: Tidak ada poligon valid yang ditemukan di {file_basename}.")
            continue

        # 5. BUAT DAN SIMPAN JSON LABELME
        labelme_data = polygons_to_labelme_json(
            all_detections_data_for_this_image,
            image_name, 
            height,
            width,
            label="tree" # Ganti label jika perlu
        )
        
        # Simpan file LabelMe dengan NAMA FILE YANG SAMA di folder output
        output_filename = os.path.join(output_labelme_dir, file_basename)
        
        try:
            with open(output_filename, 'w') as f:
                json.dump(labelme_data, f, indent=2)
            count += 1
        except IOError as e:
            print(f"  ERROR: Gagal menyimpan file {output_filename}. Detail: {e}")

    print(f"\n✅ Selesai. Berhasil mengonversi dan menyimpan {count} file LabelMe JSON.")
    print(f"Output disimpan di: {output_labelme_dir}")

# Jalankan fungsi utama
if __name__ == "__main__":
    sys.setrecursionlimit(5000) # Untuk poligon yang kompleks
    main_individual_rle_converter()

🚀 Memulai konversi (Mode Individual)...
Mendeteksi 169 file .json di: ./sample/tiles/predictions/
Menyimpan file LabelMe ke: ./sample/tiles/
[1/169] Memproses: tile_0_0.json
[2/169] Memproses: tile_0_1.json
[3/169] Memproses: tile_0_10.json
[4/169] Memproses: tile_0_11.json
[5/169] Memproses: tile_0_12.json
[6/169] Memproses: tile_0_2.json
[7/169] Memproses: tile_0_3.json
[8/169] Memproses: tile_0_4.json
[9/169] Memproses: tile_0_5.json
[10/169] Memproses: tile_0_6.json
[11/169] Memproses: tile_0_7.json
[12/169] Memproses: tile_0_8.json
[13/169] Memproses: tile_0_9.json
[14/169] Memproses: tile_10_0.json
[15/169] Memproses: tile_10_1.json
[16/169] Memproses: tile_10_10.json
[17/169] Memproses: tile_10_11.json
[18/169] Memproses: tile_10_12.json
[19/169] Memproses: tile_10_2.json
[20/169] Memproses: tile_10_3.json
[21/169] Memproses: tile_10_4.json
[22/169] Memproses: tile_10_5.json
[23/169] Memproses: tile_10_6.json
[24/169] Memproses: tile_10_7.json
[25/169] Memproses: tile_10_8.json


# Cek Area < 1024 piksel (< APs)

## hapus area < 1024 piksel

In [11]:
imgs_dir

'./sample/tiles/'

In [9]:
import json
import glob
import os
from shapely.geometry import Polygon

def delete_small_polygons(directory_path, area_threshold=1024):
    """
    Menghapus poligon yang areanya LEBIH KECIL dari ambang batas (threshold).
    
    PERINGATAN: Skrip ini akan menimpa file JSON asli.
    (Versi ini telah memperbaiki bug perhitungan total_polygons_deleted)
    """
    json_files = glob.glob(os.path.join(directory_path, "*.json"))
    
    if not json_files:
        print(f"Peringatan: Tidak ada file .json yang ditemukan di '{directory_path}'")
        return

    print(f"Memindai {len(json_files)} file JSON di '{directory_path}'...")
    print(f"Akan menghapus poligon dengan area < {area_threshold} piksel persegi.")
    
    total_files_modified = 0
    total_polygons_deleted = 0 # Inisialisasi di 0

    for json_file_path in json_files:
        filename = os.path.basename(json_file_path)
        
        try:
            with open(json_file_path, 'r') as f:
                data = json.load(f)
            
            shapes = data.get('shapes', [])
            if not shapes:
                continue
                
            original_shape_count = len(shapes)
            new_shapes = []
            deleted_in_this_file = 0 # Counter untuk file ini

            for shape in shapes:
                if (shape.get('shape_type') != 'polygon' and shape.get('shape_type') != None):
                    new_shapes.append(shape)
                    continue
                    
                points = shape.get('points', [])
                
                if len(points) < 3:
                    total_polygons_deleted += 1 # Hitung ini sebagai terhapus
                    deleted_in_this_file += 1
                    print(f"  [File: {filename}] Menghapus shape tidak valid (titik < 3).")
                    continue 

                try:
                    poly = Polygon(points)
                    
                    if not poly.is_valid:
                        poly = poly.buffer(0)
                        
                    if poly.is_empty:
                        area = 0.0
                    else:
                        area = poly.area

                    # Logika Filter
                    if area >= area_threshold:
                        new_shapes.append(shape)
                    else:
                        # === INI ADALAH PERBAIKANNYA ===
                        total_polygons_deleted += 1 # Tambahkan ke counter total
                        deleted_in_this_file += 1   # Tambahkan ke counter file
                        # ================================
                        
                        label = shape.get('label', 'unknown_shape')
                        print(f"  [File: {filename}] Menghapus '{label}' (Area: {area:.2f} piksel)")

                except Exception as e:
                    print(f"  [Error Geometri: {filename}] Gagal memproses 1 shape: {e}. Shape asli dipertahankan.")
                    new_shapes.append(shape)
            
            # Tulis ulang file jika ada perubahan
            if len(new_shapes) < original_shape_count:
                total_files_modified += 1
                data['shapes'] = new_shapes
                
                with open(json_file_path, 'w') as f:
                    json.dump(data, f, indent=2)
                
                # Pastikan jumlahnya konsisten
                print(f"  -> {filename} dimodifikasi, {deleted_in_this_file} poligon kecil dihapus.")

        except json.JSONDecodeError:
            print(f"Error: Gagal membaca file JSON (format salah): {filename}")
        except Exception as e:
            print(f"Error saat memproses {filename}: {e}")

    print("\n--- PROSES PENGHAPUSAN AREA KECIL SELESAI ---")
    print(f"Total file yang dimodifikasi: {total_files_modified}")
    print(f"Total poligon yang dihapus:   {total_polygons_deleted}") # Sekarang harusnya benar

# --- CARA PENGGUNAAN ---
if __name__ == "__main__":
    # GANTI DENGAN PATH SEBENARNYA KE DIREKTORI ANOTASI ANDA
    # (Ingat, buat backup dulu!)
    PATH_KE_DIREKTORI_JSON = imgs_dir
    
    # Ambang batas area sesuai metrik COCO 'small'
    AREA_THRESHOLD = 1024 
    
    delete_small_polygons(PATH_KE_DIREKTORI_JSON, area_threshold=AREA_THRESHOLD)

Memindai 3412 file JSON di 'D:/DATASET/JATINANGOR/tiles/png/'...
Akan menghapus poligon dengan area < 1024 piksel persegi.
  [File: tile_10_10.json] Menghapus 'tree' (Area: 25.50 piksel)
  -> tile_10_10.json dimodifikasi, 1 poligon kecil dihapus.
  [File: tile_10_30.json] Menghapus 'tree' (Area: 33.50 piksel)
  [File: tile_10_30.json] Menghapus 'tree' (Area: 11.50 piksel)
  [File: tile_10_30.json] Menghapus 'tree' (Area: 15.50 piksel)
  [File: tile_10_30.json] Menghapus 'tree' (Area: 11.50 piksel)
  [File: tile_10_30.json] Menghapus 'tree' (Area: 7.50 piksel)
  -> tile_10_30.json dimodifikasi, 5 poligon kecil dihapus.
  [File: tile_10_31.json] Menghapus 'tree' (Area: 25.50 piksel)
  -> tile_10_31.json dimodifikasi, 1 poligon kecil dihapus.
  [File: tile_10_33.json] Menghapus 'tree' (Area: 14.50 piksel)
  -> tile_10_33.json dimodifikasi, 1 poligon kecil dihapus.
  [File: tile_10_35.json] Menghapus 'tree' (Area: 9.50 piksel)
  -> tile_10_35.json dimodifikasi, 1 poligon kecil dihapus.
  [